# 🚲 Bicycle RTI Hackathon — One-Click Deployment

This notebook deploys the **complete** Bicycle Real-Time Intelligence solution into your workspace.

## What Gets Deployed (26 items)
| Stage | Items |
|-------|-------|
| 1. Storage | 4 Lakehouses |
| 2. Real-Time | 1 Eventhouse → 1 KQL Database (depends on Eventhouse) |
| 3. Compute & Ingestion | 9 Notebooks, 2 Eventstreams |
| 4. Analytics | 2 Semantic Models, 1 Pipeline |
| 5. Presentation | 1 Report, 1 KQL Dashboard, 2 Data Agents, 2 Activators |

## Before you start (one-time setup)

1. Create a **temporary Lakehouse** in your workspace (e.g. name it `deploy_staging`)
2. Open the lakehouse → click **Upload → Upload folder**
3. Upload the **`workspace`** folder from the cloned repo
4. You should now see `Files/workspace/` in the lakehouse with ~26 sub-folders
5. **Attach** this lakehouse to this notebook (left sidebar → "Add lakehouse" → select `deploy_staging`)

## Instructions

1. Run **Cell 1** — installs `fabric-cicd` (one-time)
2. Run **Cell 2** — deploys all 26 items from the lakehouse files (no GitHub, no PAT, no auth prompts)
3. Run **Cell 3** — validates deployment
4. See **Cell 4** — post-deployment manual steps
5. **(Optional)** Delete the `deploy_staging` lakehouse after deployment

In [ ]:
# =============================================================
# CELL 1 — Install dependencies
# =============================================================
# Pip dependency warnings from nni/mlflow/datasets are harmless —
# they are pre-installed Fabric runtime packages, not ours.

%pip install fabric-cicd --quiet

# Restart Python so the newly installed package is importable
try:
    import notebookutils
    notebookutils.session.restartPython()
except NameError:
    print("⚠️  notebookutils not available — restart the kernel manually before Cell 2.")

In [ ]:
# =============================================================
# CELL 2 — Deploy all 26 items from Lakehouse
# =============================================================
# No GitHub download needed — reads from the attached lakehouse.
# Make sure you uploaded the workspace/ folder to the lakehouse
# and attached it to this notebook (see instructions above).
# =============================================================

import os
import shutil
import tempfile
import sempy.fabric as fabric
from fabric_cicd import FabricWorkspace, publish_all_items

# ── Locate workspace/ folder in the attached lakehouse ──
LAKEHOUSE_PATH = "/lakehouse/default/Files/workspace"

if not os.path.isdir(LAKEHOUSE_PATH):
    alt_paths = [
        "/lakehouse/default/Files/RTI-Hackathon-Demo/workspace",
        "/lakehouse/default/Files/RTI-Hackathon-Demo-main/workspace",
    ]
    found = None
    for p in alt_paths:
        if os.path.isdir(p):
            found = p
            break
    if found:
        LAKEHOUSE_PATH = found
    else:
        print("❌ Cannot find workspace/ folder in the attached lakehouse.")
        print("   Expected: /lakehouse/default/Files/workspace/")
        print("   Fix: Open your lakehouse → Upload → Upload folder → select the 'workspace' folder")
        raise FileNotFoundError(f"workspace/ not found at {LAKEHOUSE_PATH}")

items_in_folder = [d for d in os.listdir(LAKEHOUSE_PATH)
                   if os.path.isdir(os.path.join(LAKEHOUSE_PATH, d))]
print(f"📂 Found {len(items_in_folder)} items in {LAKEHOUSE_PATH}")

# ── Copy to a temp writable directory (fabric-cicd needs write access) ──
work_dir = tempfile.mkdtemp(prefix="rti_deploy_")
workspace_dir = os.path.join(work_dir, "workspace")
shutil.copytree(LAKEHOUSE_PATH, workspace_dir)
print(f"📋 Copied to temp directory: {workspace_dir}")

# ── Deploy in 5 stages (order matters for dependencies) ──
ws_id = fabric.get_workspace_id()
print(f"\n🚀 Deploying to workspace: {ws_id}")

stages = [
    # Stage 1: Lakehouses first (no dependencies)
    ("Stage 1/5: Storage (Lakehouses)",
     ["Lakehouse"]),
    # Stage 2: Eventhouse first, then KQL Database (KQL DB references Eventhouse logicalId)
    ("Stage 2/5: Real-Time (Eventhouse + KQL Database)",
     ["Eventhouse", "KQLDatabase"]),
    # Stage 3: Notebooks + Eventstreams (reference Lakehouses + Eventhouse)
    ("Stage 3/5: Compute & Ingestion (Notebooks, Eventstreams)",
     ["Notebook", "Eventstream"]),
    # Stage 4: Semantic Models + Pipeline (reference Lakehouses + Notebooks)
    ("Stage 4/5: Analytics (Semantic Models, Pipeline)",
     ["SemanticModel", "DataPipeline"]),
    # Stage 5: Everything else (reference Semantic Models + Lakehouses)
    ("Stage 5/5: Presentation & AI (Report, Dashboard, Agents, Activators)",
     ["Report", "KQLDashboard", "DataAgent", "Reflex"]),
]

for label, item_types in stages:
    print(f"\n{'='*60}")
    print(f"🚀 {label}")
    print(f"{'='*60}")
    ws = FabricWorkspace(
        workspace_id=ws_id,
        repository_directory=workspace_dir,
        item_type_in_scope=item_types,
    )
    publish_all_items(ws)
    print(f"   ✅ {label.split(':')[0]} complete")

# ── Cleanup ──
shutil.rmtree(work_dir, ignore_errors=True)

print(f"\n{'='*60}")
print("✅ DEPLOYMENT COMPLETE — 26 items deployed")
print(f"{'='*60}")
print("\nNext: Run Cell 3 to validate, then see Cell 4 for manual steps.")

In [ ]:
# =============================================================
# CELL 3 — Validate Deployment
# =============================================================
import sempy.fabric as fabric

ws_id = fabric.get_workspace_id()
print(f"Workspace: {ws_id}")

# List all items
items = fabric.list_items()
print(f"\nTotal items: {len(items)}")

# Group by type
from collections import Counter
type_counts = Counter(row['Type'] for _, row in items.iterrows())
print("\nItems by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

# Check critical items
expected = [
    "bicycles_gold", "bikerental_bronze_raw", "bicycles_silver", "weather_bronze_raw",
    "bikerentaleventhouse",
    "RTIbikeRental", "RTI-WeatherDemo",
    "PL_BicycleRTI_Medallion",
    "Bicycle RTI Analytics", "Bicycle Ontology Model",
    "Bicycle Fleet Intelligence Agent", "ontology data agent",
]
names = set(items['Display Name'])
print("\nCritical item check:")
for e in expected:
    status = "✅" if e in names else "❌ MISSING"
    print(f"  {status} {e}")

## Post-Deployment Manual Steps

The automated deployment handles 26 items. The following items require
manual setup because their item types are not yet supported by fabric-cicd:

### 1. Run the Pipeline (First Load)
1. Go to **PL_BicycleRTI_Medallion** pipeline
2. Click **Run** — this processes: Silver → Gold → ML → Ontology → SM Refresh
3. Wait ~15-25 min for completion

### 2. Ontology + Graph Model (Post-Deploy Notebook)
1. Upload and run `Post_Deploy_Setup.ipynb` to create:
   - Bicycle_Ontology_Model_New (Ontology)
   - Bicycle_Ontology_Model_New_graph (GraphModel)
   - Cycling-Campaign-Agent (OperationsAgent)

### 3. Verify Eventstreams
Both eventstreams should auto-start after deployment:
- **RTIbikeRental** — Bicycle sample data → Lakehouse + Eventhouse
- **RTI-WeatherDemo** — Real-time weather → Eventhouse

If not running, open each Eventstream and click **Start**.

### 4. Activator Rules
Both Reflex/Activator items are deployed as shells. Add alert rules:
- **BicycleFleet_Activator**: Low stock alert, high demand surge
- **Cycling Campaign Activator**: Campaign trigger based on weather

See `docs/ACTIVATOR_SETUP.md` in the repo for rule definitions.